# 01: Exploratory Data Analysis

**Question:** Do sellers acquired through different marketing channels perform differently in their first 90 days?

**Data:** `workspace.marts.fact_seller_channel_performance`, one row per acquired seller (667 sellers who signed by June 2, 2018).

**Goal of this notebook:** understand distributions, missing values, and outliers before running any statistical tests.

In [0]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

df = spark.table("workspace.marts.fact_seller_channel_performance").toPandas()

numeric_cols = [
    "revenue_90d", "orders_90d", "days_to_close", "days_to_first_sale",
    "avg_review_score_90d", "reviewed_orders_90d",
    "avg_delivery_delay_days_90d", "late_delivery_rate_90d",
]
df[numeric_cols] = df[numeric_cols].astype(float)

print(df.shape)
df.head()

In [0]:
summary = pd.DataFrame({
    "dtype": df.dtypes.astype(str),
    "missing": df.isna().sum(),
    "missing_pct": (df.isna().mean() * 100).round(1),
})
summary

In [0]:
channel_summary = (
    df.groupby("channel_group")
      .agg(
          sellers=("seller_id", "count"),
          active=("is_active_90d", "sum"),
          activation_rate=("is_active_90d", "mean"),
      )
      .sort_values("sellers", ascending=False)
)
channel_summary["activation_rate"] = channel_summary["activation_rate"].round(3)
channel_summary

In [0]:
active = df[df["is_active_90d"]]

print(active["revenue_90d"].describe().round(2))
print("\nSkewness:", round(active["revenue_90d"].skew(), 2))

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].hist(active["revenue_90d"], bins=40)
axes[0].set_title("90-day revenue (active sellers)")
axes[0].set_xlabel("Revenue (R$)")
axes[0].set_ylabel("Sellers")

axes[1].hist(np.log1p(active["revenue_90d"]), bins=40)
axes[1].set_title("Log of 90-day revenue")
axes[1].set_xlabel("log(1 + revenue)")

plt.tight_layout()
plt.show()

In [0]:
order = channel_summary.index.tolist()
data = [np.log1p(active.loc[active["channel_group"] == ch, "revenue_90d"]) for ch in order]

fig, ax = plt.subplots(figsize=(10, 5))
ax.boxplot(data, tick_labels=order)
ax.set_title("Log 90-day revenue by channel (active sellers)")
ax.set_ylabel("log(1 + revenue)")
plt.xticks(rotation=30)
plt.tight_layout()
plt.show()

## EDA findings

1. **Sample:** 667 sellers with a full 90-day window; 289 (43.3%) made at least one sale.
2. **Missing values are structural, not data loss.** Review, delivery, and category fields are empty for the 378 inactive sellers, since there is nothing to measure. `lead_type` (4) and `business_type` (9) have small gaps, which will be filled with "unknown" so no sellers are dropped from the regression.
3. **Revenue is extremely skewed.** Among active sellers, the median 90-day revenue is R\$324 versus a mean of R\$1,160 (skewness 12.8). The largest seller earned R\$74,434.
4. **Log revenue is roughly normal**, so the regression will model log(1 + revenue). Revenue comparisons across channels will use rank-based tests (Kruskal-Wallis) and medians rather than means.
5. **Activation varies by channel**, from 34.1% (other) to 54.8% (paid search). Whether these differences are statistically significant is tested in `02_statistical_tests`. 
6. **Revenue among active sellers looks similar across channels.** On the log scale, the channel box plots overlap heavily, with medians between roughly R\$200 and R\$400. Channel differences appear mainly in activation, not in revenue once active; this is tested formally in `02_statistical_tests`.